# <a id='toc1_'></a>[Renal Function - Infants (before year 1) | Recursive Feature Elimination - logistic regression creatinine features](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Renal Function - Infants (before year 1) | Recursive Feature Elimination - logistic regression without penalty](#toc1_)    
  - [Configuration & data](#toc1_1_)    
  - [Modelling](#toc1_2_)    
    - [Recursive feature elimination with logistic regression without penalty](#toc1_2_1_)    
  - [Results - Visualisations and significance testing](#toc1_3_)    
    - [Precision recall AUC by model feature count](#toc1_3_1_)    
    - [Anova - Precision-Recall AUC](#toc1_3_2_)    
    - [T-test - best vs 2nd best model with fewer features](#toc1_3_3_)    
    - [Balanced accuracy by model feature count](#toc1_3_4_)    
    - [Feature Coefficients of best model](#toc1_3_5_)    
    - [Confusion matrix of the best model](#toc1_3_6_)    
    - [Feature correlation heatmap of top features in the best model](#toc1_3_7_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Configuration & data](#toc0_)

In [ ]:
# logging setup
from puv.logging_setup import setup
setup()  # or DEBUG if you want more detail

import logging
logger = logging.getLogger(__name__)
logger.info("Logging initialized.")

In [ ]:
# Environment setup
from pathlib import Path

BASE = Path('/workspaces/CODESPACE/').resolve()

# Define paths for data and analysis directories & ensure key folders exist
PREP = BASE / "data" / "prep"
INT = BASE / "data" / "int" 
RES = BASE / "results" / "eda" 

for p in [PREP, INT]:
    p.mkdir(parents=True, exist_ok=True)

logger.info("Project base set to: %s", BASE)

In [ ]:
# Import dependencies
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import average_precision_score, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

from puv.modelling.impute import GowerKNNImputer

plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 15, "axes.labelsize": 13,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
})


In [ ]:
# load data for exploration
df_all = pd.read_excel(PREP / "data_all.xlsx")
df_crea = df_all[['TUR age (first)', 'Renal function at last follow up', 'Nadir creatinine', 'Baseline creatinine', 'CKD5']].copy()

# filter out non-infants
print ("before filtering: ",  len (df_crea))

# retain only patient data pertaining to infants -- i.e., less than or equal to 12 months old (based on the first TUR age)
df_crea = df_crea[df_crea['TUR age (first)'] <= 12].reset_index(drop=True)

# filter out non-infants
print ("after filtering: ",  len (df_crea))

# Create a new column 'CKD stages' based on the conditions
df_crea.loc[df_crea['CKD5'] == 1, 'CKD stages'] = '[ 5 ]'
df_crea.loc[(df_crea['CKD5'] != 1) & (df_crea['Renal function at last follow up'] == 1), 'CKD stages'] = '[ 3 - 4 ]'
df_crea.loc[(df_crea['CKD5'] != 1) & (df_crea['Renal function at last follow up'] != 1), 'CKD stages'] = '[ 1 - 2 ]'

df_nadircrea = df_crea[['Nadir creatinine', 'Renal function at last follow up', 'CKD stages']].copy().dropna()
df_baselinecrea = df_crea[['Baseline creatinine', 'Renal function at last follow up', 'CKD stages']].copy().dropna()

# Uncomment to display the value counts of CKD stages in each dataset
# display(df_crea['CKD stages'].value_counts())
# display(df_nadircrea['CKD stages'].value_counts())
# display(df_baselinecrea['CKD stages'].value_counts())   



In [ ]:
df_renal_raw = pd.read_excel(PREP / "renal_set_infants.xlsx")

# manual selection of features for the renal set -- note these are Z-scaled, unlike the ones above. 
df_renal = df_renal_raw [[  'Patient number',
                            'Nadir creatinine',
                            'Weight (1 year)',
                            'Baseline creatinine', 
                            'Bilateral hydronephrosis',
                            'No hydronephrosis',
                            'Unilateral hydronephrosis',
                            'Bilateral dysplasia',
                            'No dysplasia',
                            'Unilateral dysplasia',
                            'Urinoma and/or Acites',
                            'Thick bladder wall',
                            'Unilateral VUR',
                            'Bilateral VUR',
                            'No VUR',
                            'CKD5',
                            'Prenatal diagnosis',
                            'Rec. UTI < 1 year',
                            'Renal function at last follow up'
                            ]].copy()


# Uncomment to display the columns of the renal function dataset
# display(sorted(df_renal.columns.to_list()))

print ("Renal set length: ",  len (df_renal))


### Evaluation of standard logistic regression on the renal dataset

In [ ]:
# Impute missing values
# Missingness summary
na_counts = df_renal.isna().sum().sort_values(ascending=False)
print(na_counts)

# Gower KNN impute (mixed numeric + categorical)
imputer = GowerKNNImputer(
    n_neighbors=3,
    weights="distance",
    fallback_numeric="median",
    fallback_categorical="most_frequent",
)

target_col = "Renal function at last follow up"
if target_col in df_renal.columns:
    X = df_renal.drop(columns=[target_col])
    X_imputed = imputer.fit_transform(X)
    df_renal_imputed = df_renal.copy()
    df_renal_imputed[X.columns] = X_imputed
else:
    df_renal_imputed = imputer.fit_transform(df_renal)


In [ ]:
# Calculate PR-AUC and balanced accuracy for 5-fold stratified cross validation

# Data
X = df_renal_imputed.drop(columns=["Renal function at last follow up", "Patient number", "CKD5"])
y = df_renal_imputed["Renal function at last follow up"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

coefs = []
pr_scores = []
bacc_scores = []

for tr, te in cv.split(X, y):
    X_tr, X_te = X.iloc[tr], X.iloc[te]
    y_tr, y_te = y.iloc[tr], y.iloc[te]

    clf = LogisticRegression(max_iter=2000, class_weight=None, solver="lbfgs")
    clf.fit(X_tr, y_tr)

    # coefficients
    coefs.append(clf.coef_.ravel())

    # metrics
    if hasattr(clf, "predict_proba"):
        y_pos = clf.predict_proba(X_te)[:, 1]
    else:
        y_pos = clf.decision_function(X_te)

    pr_scores.append(average_precision_score(y_te, y_pos))
    bacc_scores.append(balanced_accuracy_score(y_te, clf.predict(X_te)))

coef_mat = np.vstack(coefs)
mean_coef = coef_mat.mean(axis=0)
sem_coef = coef_mat.std(axis=0, ddof=1) / np.sqrt(coef_mat.shape[0])

coef_df = pd.DataFrame({
    "feature": X.columns,
    "mean": mean_coef,
    "sem": sem_coef,
})

# sort by mean (signed); use abs() if you want magnitude ranking
coef_df = coef_df.sort_values("mean").reset_index(drop=True)

plt.figure(figsize=(8, max(4, 0.35 * len(coef_df))))
plt.barh(
    coef_df["feature"],
    coef_df["mean"],
    xerr=coef_df["sem"],
    color="skyblue",
    edgecolor="black",
    capsize=4
)

# jittered fold-level points
jitter = 0.12
for i, feat in enumerate(coef_df["feature"]):
    raw_vals = coef_mat[:, list(X.columns).index(feat)]
    y_pos = i + np.random.uniform(-jitter, jitter, size=raw_vals.shape[0])
    plt.scatter(raw_vals, y_pos, color="grey", s=20, alpha=0.65, zorder=3)

plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Coefficient (mean ± SEM)")
plt.title("Logistic Regression Coefficients Across CV Folds")
plt.tight_layout()
plt.show()

pr_scores = np.array(pr_scores)
bacc_scores = np.array(bacc_scores)

pr_sem = pr_scores.std(ddof=1) / np.sqrt(pr_scores.size)
bacc_sem = bacc_scores.std(ddof=1) / np.sqrt(bacc_scores.size)

print("Mean PR-AUC:", pr_scores.mean().round(3), "Std error:", pr_sem.round(3))
print("Mean balanced accuracy:", bacc_scores.mean().round(3), "Std error:", bacc_sem.round(3))


### Visualisation of logistic regression based on Nadir Creatinine

In [ ]:
# Define the features and target variable

X = df_nadircrea[['Nadir creatinine']]
y = df_nadircrea["Renal function at last follow up"]

logreg = LogisticRegression(max_iter=2000, class_weight=None)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    logreg,
    X,
    y,
    cv=cv,
    scoring={
        "pr_auc": "average_precision",       
        "balanced_accuracy": "balanced_accuracy",  
    },
    return_train_score=False,
)

pr_scores = cv_results["test_pr_auc"]
bacc_scores = cv_results["test_balanced_accuracy"]

# print("PR-AUC per fold:", pr_scores)
print("Mean PR-AUC:", pr_scores.mean().round(3), "Std error:", (pr_scores.std(ddof=1)/np.sqrt(pr_scores.size)).round(3))
print()

# print("Balanced accuracy per fold:", bacc_scores)
print("Mean balanced accuracy:", bacc_scores.mean().round(3), "Std error:", (bacc_scores.std(ddof=1)/np.sqrt(bacc_scores.size)).round(3))



In [ ]:
# Visualisation of decision boundary with logistic regression

# Train logistic regression using only the selected feature
logreg = LogisticRegression(max_iter=2000, class_weight=None)
logreg.fit(X[["Nadir creatinine"]], y) # Train on Nadir creatinine 

# Generate a range of values for Nadir creatinine (for visualization)
x_range = np.linspace(X["Nadir creatinine"].min() - 1, X["Nadir creatinine"].max() + 1, 1000)
x_range_df = pd.DataFrame(x_range, columns=["Nadir creatinine"])  # Ensure DataFrame matches feature names

# Predict probabilities
y_prob = logreg.predict_proba(x_range_df)[:, 1]  # Get probability for class 1 --> Renal function at last follow up = 1

# Find the decision boundary (where probability = 0.5)
decision_boundary = x_range[np.abs(y_prob - 0.5).argmin()]  # Closest x value where probability is 0.5

guidance_boundary_1 = 31
guidance_boundary_2 = 75

# Define jitter strength
jitter_strength = 0.02

# Jitter around the binary target
y_jittered = y + np.random.normal(0, jitter_strength, size=y.shape)

# Plot the decision boundary
plt.figure(figsize=(16, 8))

# Define the category order in ascending order
category_order = sorted(df_nadircrea['CKD stages'].unique())

# Define a color map for the labels
color_map = {
    '[ 5 ]': '#FF0000',  # Red
    '[ 3 - 4 ]': '#FFB6C1',  # Light red/pink
    '[ 1 - 2 ]': '#ADD8E6'  # Light blue
}

# Define the category order in ascending order
category_order = sorted(df_nadircrea['CKD stages'].unique())


for label in category_order:
    mask = df_nadircrea["CKD stages"] == label
    plt.scatter(
        X.loc[mask, "Nadir creatinine"],
        y_jittered[mask],
        color=color_map[label],
        label=f"CKD stages: {label}",
        edgecolor="k",
        alpha=0.8,
    )

plt.plot(x_range, y_prob, color='red', label="Logistic Regression Curve")
plt.axhline(0.5, color='lightgray', linestyle='--', label="Probability = 0.5")
plt.axvline(decision_boundary, color='black', linestyle='dashed', label=f"Decision Boundary ({decision_boundary:.2f})")
plt.axvline(guidance_boundary_1, color='gray', linestyle='dashed', label=f"Guidance Boundary A ({guidance_boundary_1:.2f})")
plt.axvline(guidance_boundary_2, color='gray', linestyle='dashed', label=f"Guidance Boundary B ({guidance_boundary_2:.2f})")


plt.xlim([0, 280])  # Set x-axis limits
plt.xticks(np.arange(0, 290, 10))
# Labels & legend
plt.xlabel("Nadir creatinine [μmol/L]")  
plt.ylabel("Predicted Probability")
plt.suptitle("Prediction of kidney failure based on nadir creatinine level", fontsize=22)
plt.title("Logistic Regression Visualization with Decision Boundary")
plt.legend()
plt.savefig(RES / "logistic_regression_nadir_creatinine_decision_boundary.png", dpi=300)
plt.show()


### Visualisation of logistic regression based on Baseline Creatinine

In [ ]:
# Define the features and target variable

X = df_baselinecrea[['Baseline creatinine']]
y = df_baselinecrea["Renal function at last follow up"]

logreg = LogisticRegression(max_iter=2000, class_weight=None)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = cross_validate(
    logreg,
    X,
    y,
    cv=cv,
    scoring={
        "pr_auc": "average_precision",       # main metric you already use
        "balanced_accuracy": "balanced_accuracy",  # extra metric to report
    },
    return_train_score=False,
)

pr_scores = cv_results["test_pr_auc"]
bacc_scores = cv_results["test_balanced_accuracy"]

# print("PR-AUC per fold:", pr_scores)
print("Mean PR-AUC:", pr_scores.mean().round(3), "Std eror:", (pr_scores.std(ddof=1)/np.sqrt(pr_scores.size)).round(3))
print()

# print("Balanced accuracy per fold:", bacc_scores)
print("Mean balanced accuracy:", bacc_scores.mean().round(3), "Std error:", (bacc_scores.std(ddof=1)/np.sqrt(bacc_scores.size)).round(3))



In [ ]:
# Visualisation of decision boundary with logistic regression

# Train logistic regression using only the selected feature
logreg = LogisticRegression(max_iter=2000, class_weight=None)
logreg.fit(X[["Baseline creatinine"]], y) # Train on Baseline creatinine 

# Generate a range of values for Baseline creatinine (for visualization)
x_range = np.linspace(X["Baseline creatinine"].min() - 1, X["Baseline creatinine"].max() + 1, 1000)
x_range_df = pd.DataFrame(x_range, columns=["Baseline creatinine"])  # Ensure DataFrame matches feature names

# Predict probabilities
y_prob = logreg.predict_proba(x_range_df)[:, 1]  # Get probability for class 1 --> Renal function at last follow up = 1

# Find the decision boundary (where probability = 0.5)
decision_boundary = x_range[np.abs(y_prob - 0.5).argmin()]  # Closest x value where probability is 0.5

# Define jitter strength
jitter_strength = 0.02

# Add jitter to the y-values of the scatter plot (keeping points around 0 or 1)
y_jittered = y + np.random.normal(0, jitter_strength, size=y.shape)

# Plot the decision boundary
plt.figure(figsize=(16, 8))

# Define the category order in ascending order
category_order = sorted(df_baselinecrea['CKD stages'].unique())

# Define a color map for the labels
color_map = {
    '[ 5 ]': '#FF0000',  # Red
    '[ 3 - 4 ]': '#FFB6C1',  # Light red/pink
    '[ 1 - 2 ]': '#ADD8E6'  # Light blue
}

# Define the category order in ascending order
category_order = sorted(df_baselinecrea['CKD stages'].unique())

for label in category_order:
    mask = df_baselinecrea["CKD stages"] == label
    plt.scatter(
        X.loc[mask, "Baseline creatinine"],
        y_jittered[mask],
        color=color_map[label],
        label=f"CKD stages: {label}",
        edgecolor="k",
        alpha=0.8,
    )


plt.plot(x_range, y_prob, color='red', label="Logistic Regression Curve")
plt.axhline(0.5, color='lightgray', linestyle='--', label="Probability = 0.5")
plt.axvline(decision_boundary, color='black', linestyle='dashed', label=f"Decision Boundary ({decision_boundary:.2f})")

plt.xlim([0, 500])  # Set x-axis limits
plt.xticks(np.arange(0, 520, 20))
# Labels & legend
plt.xlabel("Baseline creatinine [μmol/L]")  
plt.ylabel("Predicted Probability")
plt.suptitle("Prediction of kidney failure based on baseline creatinine level", fontsize=22)
plt.title("Logistic Regression Visualization with Decision Boundary")
plt.legend()
plt.savefig(RES / "logistic_regression_baseline_creatinine_decision_boundary.png", dpi=300)
plt.show()
